## Login

In [2]:
from huggingface_hub import login, whoami
login()

try:
    user_info = whoami()
    print(f"✅ Zalogowano jako: {user_info['name']}")
except Exception:
    print("❌ Logowanie nie przebiegło pomyślnie")

✅ Zalogowano jako: MorphologicalComputations


## GPU batching

The `compute_log_probabilities_for_batch` calculates log probabilities for given batch size (how surprising…), and the number of tokens for each sentence. In he `Validation` section I show that this method is compatible with other libraries

The function returs `token_counts` because it is a confound. It's worth noting though that [TurBLIMP paper (appendix E)](https://aclanthology.org/2025.emnlp-main.834.pdf#page=15) found that they don't confound much (small $R^2$). Their result is very weird, because it is a confound

### Groundwork

In [3]:
# Libraries and functions
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd

def compute_log_probabilities_for_batch(input_texts):
    inputs = tokenizer(input_texts, 
                       return_tensors="pt", 
                       padding=True, 
                       truncation=True)
    
    input_ids = inputs["input_ids"].to(device)
    attention_mask = inputs["attention_mask"].to(device)
        
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
    shift_logits = logits[:, :-1, :]
    shift_labels = input_ids[:, 1:]
    log_probs = torch.nn.functional.log_softmax(shift_logits, dim=-1)
    target_log_probs = log_probs.gather(dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1)
    target_log_probs = target_log_probs * attention_mask[:, 1:].to(log_probs.dtype)
    log_likelihood = target_log_probs.sum(dim=-1)
    return {
        "log_probabilities": log_likelihood,
        "token_counts": attention_mask[:, 1:].sum(dim=-1).tolist()
    }

if torch.cuda.is_available():
    device = 'cuda' # setting GPU, so that the code runs faster…
    print('GPU is available!')
else:
    print('GPU is NOT available!!!')

GPU is available!


Models to run:
- "speakleash/Bielik-1.5B-v3"
- "Qwen/Qwen2.5-14B-Instruct"

In [4]:
# Loading model — this might take long
MODEL = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL).to(device)

# This tells the tokenizer to reuse the end-of-sequence token (`eos_token`) as the padding token (`pad_token`)
tokenizer.pad_token = tokenizer.eos_token
print("Loading complete")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loading complete


### Main function

In [9]:
import os

# Constants
INPUT_FOLDER = "/kaggle/input/datasets/antoniustrzycki/real-real/final_cleaned/final_cleaned"
OUTPUT_FOLDER = "/kaggle/working"
BATCH_SIZE = 32
TO_SKIP = {'clean_subj_adjectival_number.csv', 'clean_subj_verb_person_simple.csv', 'clean_subj_verb_gender_simple.csv'}

           
_, model_name = MODEL.split("/")


# Collect all CSV files in the input folder
input_file_paths = []
for file_name in os.listdir(INPUT_FOLDER):
    if file_name in TO_SKIP:
        continue
    if file_name.endswith(".csv"):
        input_file_paths.append(os.path.join(INPUT_FOLDER, file_name))

# Process each file
for input_file_path in input_file_paths:

    file_name = os.path.basename(input_file_path)
    output_file_path = os.path.join(OUTPUT_FOLDER, model_name, '_', file_name)
    print(f"\nProcessing: {file_name}")

    # Load input data
    dataframe = pd.read_csv(input_file_path)
    syntactic_sentence = dataframe["syntactic_sentence"].tolist()
    unsyntactic_sentence = dataframe["unsyntactic_sentence"].tolist()
    conllu_index = dataframe["conllu_index"].tolist()
    dataset = dataframe["dataset"].tolist()

    # Process sentences in batches and compute perplexity scores
    syntactic_sentence_probabilities = []
    unsyntactic_sentence_probabilities = []
    syntactic_sentence_nr_of_tokens = []
    unsyntactic_sentence_nr_of_tokens = []

    for batch_start in range(0, len(syntactic_sentence), BATCH_SIZE):
        syntactic_sentence_batch = syntactic_sentence[batch_start : batch_start + BATCH_SIZE]
        unsyntactic_sentence_batch = unsyntactic_sentence[batch_start : batch_start + BATCH_SIZE]
        syntactic_sentence_results = compute_log_probabilities_for_batch(syntactic_sentence_batch)
        unsyntactic_sentence_results = compute_log_probabilities_for_batch(unsyntactic_sentence_batch)
        syntactic_sentence_probabilities.extend(syntactic_sentence_results["log_probabilities"].tolist())
        unsyntactic_sentence_probabilities.extend(unsyntactic_sentence_results["log_probabilities"].tolist())
        syntactic_sentence_nr_of_tokens.extend(syntactic_sentence_results["token_counts"])
        unsyntactic_sentence_nr_of_tokens.extend(unsyntactic_sentence_results["token_counts"])
        processed_count = min(batch_start + BATCH_SIZE, len(syntactic_sentence))
        print(f"  {processed_count}/{len(syntactic_sentence)} sentences")

    # Export results to the output folder, keeping the original file name
    output_dataframe = pd.DataFrame(
        {
            "dataset": dataset,
            "model": [model_name] * len(dataset), # as `MODEL = "speakleash/Bielik-1.5B-v3"`
            "conllu_index": conllu_index,
            "syntactic_log_probability": syntactic_sentence_probabilities,
            "unsyntactic_log_probability": unsyntactic_sentence_probabilities,
            "syntactic_nr_of_tokens": syntactic_sentence_nr_of_tokens,
            "unsyntactic_nr_of_tokens": unsyntactic_sentence_nr_of_tokens,
            "syntactic_sentence": syntactic_sentence,
            "unsyntactic_sentence": unsyntactic_sentence,
            "syntax_type": os.path.splitext(file_name)[0]
        }
    )
    output_dataframe.to_csv(output_file_path, index=False)
    print(f"Saved: {output_file_path}")


Processing: clean_subj_verb_gender_csubj.csv
  32/73 sentences
  64/73 sentences
  73/73 sentences
Saved: /kaggle/working/clean_subj_verb_gender_csubj.csv

Processing: clean_subj_verb_number_numerals.csv
  32/703 sentences
  64/703 sentences
  96/703 sentences
  128/703 sentences
  160/703 sentences
  192/703 sentences
  224/703 sentences
  256/703 sentences
  288/703 sentences
  320/703 sentences
  352/703 sentences
  384/703 sentences
  416/703 sentences
  448/703 sentences
  480/703 sentences
  512/703 sentences
  544/703 sentences
  576/703 sentences
  608/703 sentences
  640/703 sentences
  672/703 sentences
  703/703 sentences
Saved: /kaggle/working/clean_subj_verb_number_numerals.csv

Processing: clean_subj_verb_gender_genitive.csv
  32/57 sentences
  57/57 sentences
Saved: /kaggle/working/clean_subj_verb_gender_genitive.csv

Processing: clean_subj_adjectival_gender_cop.csv
  32/511 sentences
  64/511 sentences
  96/511 sentences
  128/511 sentences
  160/511 sentences
  192/51

## CPU heuristics

Here, just for some intution and heuristics, I consider three utterences:
1. „It was a dark and stormy night” (surely extremaly iconic sentence)
2. „Lorem ipsum”
3. „Nie słyszałem jego opowieści, ani o nim”

I compute their log-probabilities and their number of tokens after tokenizing. I do it in a very step by step way. This is not scalable by any means (although I do use batching). The only aim of the code below is a heuristic one

First, I need to import my model and set tokenization in such a way that batching is possible

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd

# Loading model — this might take long
MODEL = "speakleash/Bielik-1.5B-v3"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL)

# This tells the tokenizer to reuse the end-of-sequence token (`eos_token`) as the padding token (`pad_token`)
tokenizer.pad_token = tokenizer.eos_token
print("Loading complete")

Now, I define helper functions. Everything is carefully commented

In [ ]:
def compute_log_probabilities_for_batch(input_texts):
    # Token ids with attention_mask (padding for batching)
    inputs = tokenizer(
        input_texts, return_tensors="pt", padding=True, truncation=True
    )

    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    # Conditional predictions of the next token based on the previous ones
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits # This accesses for attribute

    # Better representation
    shift_logits = logits[:, :-1, :] # A score for every token in the vocabulary at each position 
    shift_labels = input_ids[:, 1:] # IDs of tokens from batch input

    # Applies softmax function to each element in the array and gets log-probabilities for each token in the vocabulary
    log_probs = torch.nn.functional.log_softmax(shift_logits, dim=-1)
    target_log_probs = log_probs.gather(dim=-1, index=shift_labels.unsqueeze(-1)).squeeze(-1)
    
    # Multipling by the mask for batching
    target_log_probs = target_log_probs * attention_mask[:, 1:].to(log_probs.dtype)

    # My beloved log probabilities
    log_likelihood = target_log_probs.sum(dim=-1) # Just summing without taking the average

    return {
        "log_probabilities": log_likelihood,
        "token_counts": attention_mask[:, 1:].sum(dim=-1).tolist()
    }
    
print('Done')

Here are some examplary computations

In [ ]:
texts = ['It was a dark and stormy night',
        'Lorem ipsum',
        'Nie słyszałem jego opowieści, ani o nim']

z = compute_log_probabilities_for_batch(texts)

for key, val in z.items():
    print(f'key: {key}\nvalue: {val}')

Here there are step by step results for the log-probabilities for 'cat'



**shift logits**

tensor([[[-6.8125,  6.6562,  6.2188,  ...,  1.5391,  5.9375,  4.4688],
         [-5.5312,  0.9141,  7.9375,  ...,  1.7031,  9.6875,  5.4375]]],
       dtype=torch.bfloat16)

**shift_labels**

tensor([[407, 305]])

**log_probs**

tensor([[[-25.6250, -12.1875, -12.6250,  ..., -17.2500, -12.8750, -14.3750],
         [-21.1250, -14.6875,  -7.6250,  ..., -13.8750,  -5.8750, -10.1250]]],
       dtype=torch.bfloat16)
       
**target_log_probs (1)**

tensor([[-12.2500,  -3.0781]], dtype=torch.bfloat16)
       
**target_log_probs (2)**

tensor([[-12.2500,  -3.0781]], dtype=torch.bfloat16)

**log_likelihood**

tensor([-7.6562], dtype=torch.bfloat16)

**TOKENS**

attention_mask
tensor([[1, 1, 1]])

In [ ]:
# Some intuiton for the nr of tokens
import torch

input_ids = torch.tensor([
    [1, 407, 305,   0,   0],  # "cat"
    [1, 301, 402, 156, 789],  # "elephant"
])

attention_mask = torch.tensor([
    [1, 1, 1, 0, 0],  # cat: 3 real tokens, 2 padding
    [1, 1, 1, 1, 1],  # elephant: 5 real tokens
])

print(attention_mask.sum(dim=-1).tolist())         # [3, 5]
print(attention_mask[:, 1:].sum(dim=-1).tolist())  # [2, 4]

## Original TurBLIMP

In [ ]:
from minicons import scorer
import torch
import numpy as np
import csv
import os

def load_sentences(filepath):
    sentence_pairs = []
    with open(filepath, 'r', encoding='utf-8') as file:
        reader = csv.reader(file, delimiter=';')
        next(reader)
        for row in reader:
            good_sentence = row[0]
            bad_sentence = row[1]
            sentence_pairs.append([good_sentence, bad_sentence])
    return sentence_pairs

def compute_score(data, model, mode):
    if mode == 'ilm':
        score = model.sequence_score(data, reduction=lambda x: x.sum(0).item())
    elif mode == 'mlm':
        score = model.sequence_score(data, reduction=lambda x: x.sum(0).item(), PLL_metric='within_word_l2r')
    return score

def process_files(model, mode, model_name, output_folder):
    file_names = ["argument_structure_ditransitive_baseline.csv",
              "argument_structure_ditransitive_ken.csv",
              "argument_structure_ditransitive_dik.csv",
              "argument_structure_ditransitive_inca.csv",
              "argument_structure_ditransitive_OSV.csv",
              "argument_structure_ditransitive_OVS.csv",
              "argument_structure_ditransitive_finite.csv",
              "argument_structure_ditransitive_SOV.csv",
              "argument_structure_ditransitive_SVO.csv",
              "argument_structure_ditransitive_VOS.csv",
              "argument_structure_ditransitive_VSO.csv",
              "argument_structure_transitive_baseline.csv",
              "argument_structure_transitive_ken.csv",
              "argument_structure_transitive_dik.csv",
              "argument_structure_transitive_inca.csv",
              "argument_structure_transitive_OSV.csv",
              "argument_structure_transitive_OVS.csv",
              "argument_structure_transitive_finite.csv",
              "argument_structure_transitive_SOV.csv",
              "argument_structure_transitive_SVO.csv",
              "argument_structure_transitive_VOS.csv",
              "argument_structure_transitive_VSO.csv"]

    os.makedirs(output_folder, exist_ok=True)

    for file_path in file_names:
        try:
            pairs = load_sentences(file_path)
            results = []
            differences = 0
            accuracy = 0

            for pair in pairs:
                score = compute_score(pair, model, mode)
                results.append({
                    'good_sentence': pair[0],
                    'bad_sentence': pair[1],
                    'good_score': score[0],
                    'bad_score': score[1],
                    'difference': score[0] - score[1],
                    'syntactic_sentence': score[0] > score[1]
                })

                if score[0] > score[1]:
                    accuracy += 1
                differences += score[0] - score[1]

            mean_difference = differences / len(pairs)
            accuracy = accuracy / len(pairs)

            summary = {
                'file_name': file_path,
                'mean_difference': mean_difference,
                'accuracy': accuracy,
                'total_pairs': len(pairs)
            }

            output_file = os.path.join(output_folder, f"{model_name}_{file_path}")
            with open(output_file, 'w', encoding='utf-8', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=results[0].keys())
                writer.writeheader()
                writer.writerows(results)

            print(f"Processed {file_path}:")
            print(f"  Mean difference: {mean_difference:.4f}")
            print(f"  Accuracy: {accuracy:.4f}")

        except Exception as e:
            print(f"Error processing {file_path}: {str(e)}")
            continue

ilm_model_names = ['meta-llama/Llama-3.1-8B',
                   'google/gemma-2-9b',
                   'CohereForAI/aya-expanse-8b',
                   'Qwen/Qwen2.5-7B',
                   'utter-project/EuroLLM-9B',
                   'ytu-ce-cosmos/turkish-gpt2-large',
                   'goldfish-models/tur_latn_1000mb',
                   'goldfish-models/tur_latn_100mb',
                   'goldfish-models/tur_latn_10mb',
                   'goldfish-models/tur_latn_5mb',
                   'google/gemma-3-4b-pt',
                   'google/gemma-3-12b-pt']

mlm_model_names = ['dbmdz/bert-base-turkish-128k-uncased']


device = 'cuda' if torch.cuda.is_available() else 'cpu'

mode = 'mlm'
model_name = mlm_model_names[0]
model = scorer.MaskedLMScorer(model_name, device)

process_files(
    model=model,
    mode=mode,
    model_name=model_name,
    output_folder='/scores'
)

## Better TurBLIMP

In [ ]:
from minicons import scorer
import torch
import numpy as np
import csv
import os
print('Done!')

In [ ]:
def load_sentences(filepath):
    sentence_pairs = []
    with open(filepath, 'r', encoding='utf-8') as file:
        reader = csv.reader(file, delimiter=';')
        next(reader)
        for row in reader:
            good_sentence = row[0]
            bad_sentence = row[1]
            sentence_pairs.append([good_sentence, bad_sentence])
    return sentence_pairs

def compute_score(data, model, mode):
    if mode == 'ilm': # Incremental Language Model (left to right)
        score = model.sequence_score(data, reduction=lambda x: x.sum(0).item())
    elif mode == 'mlm': # Masked Language Model (biconditional)
        score = model.sequence_score(data, reduction=lambda x: x.sum(0).item(), PLL_metric='within_word_l2r')
    return score

def process_files(model, mode, model_name, output_folder, file_names):
    os.makedirs(output_folder, exist_ok=True)

    for file_path in file_names:
        try:
            pairs = load_sentences(file_path)
            results = []
            differences = 0
            accuracy = 0

            for pair in pairs:
                score = compute_score(pair, model, mode)
                results.append({
                    'good_sentence': pair[0],
                    'bad_sentence': pair[1],
                    'good_score': score[0],
                    'bad_score': score[1],
                    'difference': score[0] - score[1],
                    'syntactic_sentence': score[0] > score[1]
                })

                if score[0] > score[1]:
                    accuracy += 1
                differences += score[0] - score[1]

            mean_difference = differences / len(pairs)
            accuracy = accuracy / len(pairs)

            summary = {
                'file_name': file_path,
                'mean_difference': mean_difference,
                'accuracy': accuracy,
                'total_pairs': len(pairs)
            }

            output_file = os.path.join(output_folder, f"{model_name}_{file_path}")
            with open(output_file, 'w', encoding='utf-8', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=results[0].keys())
                writer.writeheader()
                writer.writerows(results)

            print(f"Processed {file_path}:")
            print(f"  Mean difference: {mean_difference:.4f}")
            print(f"  Accuracy: {accuracy:.4f}")

        except Exception as e:
            print(f"Error processing {file_path}: {str(e)}")
            continue
            
print('Completed')

In [ ]:
ilm_model_names = ['meta-llama/Llama-3.1-8B',
                   'google/gemma-2-9b']

mlm_model_names = ['dbmdz/bert-base-turkish-128k-uncased']

file_names = ["argument_structure_ditransitive_baseline.csv",
              "argument_structure_ditransitive_ken.csv"]

device = 'cuda' if torch.cuda.is_available() else 'cpu'

mode = 'mlm'
model_name = mlm_model_names[0]
model = scorer.MaskedLMScorer(model_name, device)

process_files(
    model=model,
    mode=mode,
    model_name=model_name,
    output_folder='/scores',
    file_names = file_names
)

## Validation

Here I validate my log-probabilities

In [ ]:
from minicons import scorer
ilm_model = scorer.IncrementalLMScorer("speakleash/Bielik-1.5B-v3")

In [ ]:
# Official computation from TurBLIMP
def compute_score(data, model, mode):
    if mode == 'ilm':
        score = model.sequence_score(data, reduction=lambda x: x.sum(0).item())
    elif mode == 'mlm':
        score = model.sequence_score(data, reduction=lambda x: x.sum(0).item(), PLL_metric='within_word_l2r')
    return score

In [ ]:
texts = ['It was a dark and stormy night',
         'Lorem ipsum',
         'Jan jest ambitny i głodny',
         'Jan jest lewicowcem i z tego dumny',
         'Daniel stał się filantropem i mądry',
         'Ona sięgnęła gwiazd i po kubek z wodą',
         'To, że się spóźniał i krzyczał na współpracowników spowodowało jego zwolnienie',
         'Nie słyszałem jego opowieści, ani o nim']

compute_score(texts, ilm_model, mode='ilm')

In [ ]:
# My computation
compute_log_probabilities_for_batch(texts)

### Nr of tokens

Results are off by 1 because of BOS

In [ ]:
texts = ['It was a dark and stormy night',
         'Lorem ipsum',
         'Jan jest ambitny i głodny',
         'Jan jest lewicowcem i z tego dumny',
         'Daniel stał się filantropem i mądry',
         'Ona sięgnęła gwiazd i po kubek z wodą',
         'To, że się spóźniał i krzyczał na współpracowników spowodowało jego zwolnienie',
         'Nie słyszałem jego opowieści, ani o nim']



token_counts = [len(tokenizer.encode(text)) for text in texts]
print(f"Token counts: {token_counts}")

In [ ]:
text = ['It was a dark and stormy night']

tokenizer.encode(text)